# Background Subtraction

A long-slit frame records the source and the sky on the same pixels. The slit is open along its
whole length, so every column contains the spectrum of the night sky (airglow, scattered
moonlight, unresolved city light) with the object's spectrum sitting on top of it in a handful
of rows. Extracting the object means subtracting everything underneath it first.

The sky is not a small correction, and more importantly it is not a *constant* one. In a
five-pixel aperture on this frame the sky typically accounts for about two percent of the flux,
but it ranges from **under 1 percent to nearly 9 percent** depending on wavelength,
because the sky spectrum has bright emission lines where the stellar spectrum does not.

`specreduce.background.Background` estimates that sky from regions of the slit *beside* the
source and removes it. The class itself is simple. It assumes that the sky in each column is a single
number, so everything inside the estimating regions gets averaged into that number. Most of
the care in using it goes into choosing those regions, and that is what this chapter is about.

In [ ]:
import sys

sys.path.append("../src")

import numpy as np
import matplotlib.pyplot as plt
from astropy.modeling import models
from astropy.visualization import simple_norm
from specreduce.background import Background
from specreduce.tilt_solution import TiltSolution
from specreduce.tracing import FitTrace

from common import TILT_SOLUTION_FILE, fit_tilt_solution, read_data

plt.rc("figure", figsize=(8, 2.4), dpi=110)
plt.rc("font", size=9)

arcs, lamps, obj = read_data()
ny, nx = obj.data.shape
rows = np.arange(ny)
columns = np.arange(nx)

Two things have to be in place before the sky can be measured.

The frames are **rectified**. The sky windows sit tens of rows away from the source, and their
columns have to hold the same wavelengths as the source's columns do. Whether that calls for a
rectification step depends on the instrument. The OSIRIS frames used here show a strong tilt
distortion, as Chapter 6 showed, so for this dataset it does. The tilt solution is not refitted
here: Chapter 6 fitted it once and wrote it to an ASDF file, and every chapter from this one on
reads it back with `TiltSolution.from_asdf`.

The guard below refits and writes the solution if the file is not there, so that this chapter
still runs on its own if you open it without having run Chapter 6.

The source is then **traced** as Chapter 4 established, on the rectified frame, because that is
the frame everything from here on is measured from.

In [ ]:
if not TILT_SOLUTION_FILE.exists():  # normally written by Chapter 6
    fit_tilt_solution(arcs).to_asdf(TILT_SOLUTION_FILE)

solution = TiltSolution.from_asdf(TILT_SOLUTION_FILE)

image = solution.resample(obj)

trace = FitTrace(
    image, bins=32, peak_method="gaussian", trace_model=models.Chebyshev1D(4)
)
trace_y = np.asarray(trace.trace)

## Reading the slit

Before placing any window, look at what is actually on the slit. The median along the dispersion
axis gives the cross-dispersion profile again, but this time the interesting part is not the
target but everything else.

In [ ]:
slit = np.median(image.data, axis=1)

fig, ax = plt.subplots(figsize=(8, 3.0), constrained_layout=True)
ax.plot(rows, slit, lw=0.8)

for row, name in (
    (135, "target"),
    (117, "faint star"),
    (36, "faint star"),
    (311, "faint star"),
):
    ax.annotate(
        name,
        (row, slit[row]),
        (row, slit[row] * 1.5),
        size=8,
        ha="center",
        arrowprops=dict(arrowstyle="-", lw=0.6, color="0.4"),
    )

ax.set(
    yscale="log",
    xlim=(0, ny - 1),
    ylim=(30, 2e5),
    xlabel="Cross-dispersion [pix]",
    ylabel="Median signal [e$^-$]",
);

Three features matter for what follows.

**The sky is flat.** The median sky level sits near 680 e⁻ along the whole slit, varying smoothly
by a few percent. Every row is sky, so a background window can go wherever the sources allow.

**There are four sources on this slit, not two.** Chapter 4 used the target at row 135 and the
faint star at row 311. There are two more, at rows **117** and **36**, both faint enough to miss
on a first look and both bright enough to poison a sky estimate. The one at 117 is only 18 rows
from the target.

**The target has wings.** The profile does not drop to sky level immediately either side of row
135. Quantifying that is the first thing we need, because it sets the minimum usable separation.

Ten rows from the trace the target still adds nearly half the sky level, even though that is
under one percent of its peak, and its wings do not fall below one percent of the sky until
about 60 rows out; the bump at 15 to 20 rows is the faint star at row 117. This is the central
tension of the chapter: windows must be **far enough** from the trace to escape its wings, and
**near enough** to stay on clean sky between the other sources.

## How `Background` works

A `Background` is defined by one or more **windows**: strips that run parallel to a trace,
displaced from it along the cross-dispersion axis. Each window is set by a `separation` (the
offset of its center from the trace) and a `width` (its extent across the slit). Because the
windows follow the trace rather than the pixel grid, they stay the same distance from the source
even as it curves, which is exactly what `trace + delta` did at the end of Chapter 4.

Two constructors cover almost every case:

- `Background.two_sided(image, trace, separation, width=...)` places two windows, one either
  side of the trace at ±`separation`.
- `Background.one_sided(image, trace, separation, width=...)` places a single window, on the side
  given by the sign of `separation`.

For each column, `Background` reduces the pixels inside the windows to **one number**, an
average or a median, and that number is the sky for every row of that column. This is the
assumption to keep in view: the model of the sky across the slit is a constant. It cannot
represent a gradient, it cannot represent a source you left in the window, and it will not warn
you about either.

In [ ]:
separation, width = 60, 20
bkg = Background.two_sided(image, trace, separation=separation, width=width)

bkg_image = bkg.bkg_image()
sub_image = bkg.sub_image()
bkg_spectrum = bkg.bkg_spectrum()

print(f"bkg_image     {bkg_image.flux.shape}  {bkg_image.flux.unit}")
print(
    f"sub_image     {sub_image.flux.shape}  uncertainty: "
    f"{type(sub_image.uncertainty).__name__}"
)
print(
    f"bkg_spectrum  {bkg_spectrum.flux.shape}  uncertainty: "
    f"{type(bkg_spectrum.uncertainty).__name__}"
)

All three are `specutils` `Spectrum` objects, the containers Chapter 3 introduced, and all three
carry uncertainties. `Background` propagates the input variance through the averaging rather than
discarding it, which matters for Chapter 8: optimal extraction weights pixels by their
uncertainty, and a background step that dropped it would silently break that weighting.

Here is the window geometry drawn on the frame, together with the result.

In [ ]:
def show_windows(ax, separation, width, ylim=(10, 260), vmax=250_000):
    """Draw the frame with the trace and the two background windows marked."""
    ax.imshow(
        image.data,
        origin="lower",
        aspect="auto",
        cmap=plt.cm.Blues,
        norm=simple_norm(image.data, stretch="log", vmin=0, vmax=vmax),
    )
    ax.plot(trace_y, c="C1", lw=1.0)
    for sign in (1, -1):
        center = trace_y + sign * separation
        ax.fill_between(
            columns,
            center - width / 2,
            center + width / 2,
            color="C3",
            alpha=0.25,
            lw=0,
        )
        ax.plot(center, c="C3", lw=0.6, ls="--")
    ax.set(xlim=(0, nx - 1), ylim=ylim, ylabel="Cross-dispersion [pix]")


fig, axs = plt.subplots(3, figsize=(8, 5.4), sharex="all", constrained_layout=True)

show_windows(axs[0], separation, width)
axs[0].set_title(
    f"rectified frame, windows at $\\pm${separation} px, width {width}", size=9
)

for ax, panel, label in (
    (axs[1], bkg_image.flux.value, "background model"),
    (axs[2], sub_image.flux.value, "background subtracted"),
):
    ax.imshow(
        panel,
        origin="lower",
        aspect="auto",
        cmap=plt.cm.Blues,
        norm=simple_norm(panel, stretch="log", vmin=0, vmax=250_000),
    )
    ax.text(0.01, 0.9, label, transform=ax.transAxes, va="top", size=8)
    ax.set(ylim=(10, 260), ylabel="Cross-dispersion [pix]")

axs[-1].set_xlabel("Dispersion [pix]");

The middle panel shows what "one constant per column" means: the background model is banded
vertically, identical down every column, varying only along the dispersion axis. Its bright
vertical stripes are sky emission lines. In the bottom panel those stripes are gone and the
diffuse glow around the trace has flattened out.

The background spectrum on its own is the sky as `Background` sees it:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.6), constrained_layout=True)
ax.plot(bkg_spectrum.flux.value, lw=0.7)
ax.set(xlim=(0, nx - 1), xlabel="Dispersion [pix]", ylabel="Sky [e$^-$ pix$^{-1}$]");

That is a recognizable night-sky spectrum: a smooth continuum with strong, narrow emission
features, the brightest near column 743. Its peak is about fifteen times its faintest point, which
is why leaving it in would be so damaging: the contamination is strongly wavelength dependent.

## Choosing the separation

Now the real work. The tension identified above, that too close catches the source and too far catches
its neighbors, means there is an optimum, and we can find it empirically.

The test is simple: subtract the background, then look at a strip of blank sky *just outside*
the upper window. If the sky estimate were perfect, that strip would average to zero. Any
departure is background that was over- or under-estimated.

In [ ]:
def residual_outside(separation, width, gap=5, depth=20):
    """Median residual in a blank strip just beyond the upper background window."""
    background = Background.two_sided(image, trace, separation=separation, width=width)
    subtracted = background.sub_image().flux.value
    low = int(round(trace_y.mean() + separation + width / 2 + gap))
    return np.median(np.median(subtracted[low : low + depth], axis=0))


separations = np.arange(25, 106, 10)
residuals = np.array([residual_outside(s, 20) for s in separations])

for s, r in zip(separations, residuals):
    print(f"separation {s:3d}:  residual {r:8.2f} e-")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.8), constrained_layout=True)
ax.plot(separations, residuals, "o-", lw=0.9, ms=4)
ax.axhline(0, c="0.6", lw=0.8, ls="--")
ax.annotate(
    "source wings\nin the window",
    (25, residuals[0]),
    (40, -70),
    size=8,
    arrowprops=dict(arrowstyle="->", lw=0.6, color="0.4"),
)
ax.annotate(
    "window reaches\nthe trace at row 36",
    (95, residuals[-1]),
    (62, -45),
    size=8,
    arrowprops=dict(arrowstyle="->", lw=0.6, color="0.4"),
)
ax.set(xlabel="Separation [pix]", ylabel="Residual sky [e$^-$]");

The curve has exactly the shape the slit profile predicted. At separation 25 the windows sit on
the source's wings, the sky is overestimated by a wide margin, and subtracting it drives the rest
of the frame 86 e⁻ negative, more than ten percent of the sky level. The residual improves
rapidly out to about 75 pixels, where it is under half an electron, and then **collapses again
beyond 85**, to −44 e⁻ at separation 95 and −34 e⁻ at 105.

That collapse is not noise. At separation 95 the lower window spans rows 30 to 50, and
the faint star at row 36 sits inside it. Its flux inflates the sky estimate just as the target's
wings did at the other end. Neither failure produced a warning; both would have quietly biased
every extracted flux.

Anything between roughly 50 and 85 is defensible here. We will use **separation 60, width 20**,
comfortably clear of the wings and avoiding all three neighboring sources, and Chapters 8 and 12
will reuse it.

:::{note}
The `width` trades noise against risk. A wider window averages more pixels, so the sky estimate
is less noisy (the photon noise falls as $1/\sqrt{N}$), but it also spans more of the slit, and
is therefore more likely to reach a neighbor or a region where the illumination changes. Twenty
rows across two windows gives 40 pixels per column here, enough that the sky estimate contributes
negligible noise to the extraction.
:::

## One-sided and two-sided windows

`two_sided` averages a window on each side of the trace. `one_sided` uses a single window, on
whichever side the sign of `separation` selects. The difference is more than convenience:

In [ ]:
options = (
    ("two_sided", Background.two_sided(image, trace, separation=60, width=20)),
    ("one_sided, above", Background.one_sided(image, trace, separation=60, width=20)),
    ("one_sided, below", Background.one_sided(image, trace, separation=-60, width=20)),
)

for label, background in options:
    flux = background.bkg_spectrum().flux.value
    print(f"{label:>18}:  sky at column 512 = {flux[512]:7.2f} e-")

The two sides disagree by about four percent, 842 e⁻ above the trace against 873 e⁻ below,
and the two-sided value, 858 e⁻, falls neatly between them. That difference is a real gradient
in the sky across the slit, from vignetting and from the slit's own illumination pattern.

This is the argument for `two_sided` wherever it is available. `Background` fits a constant, so it
cannot follow a gradient; but averaging two windows placed symmetrically about the trace
evaluates that gradient at the trace's own position, and a **linear** gradient cancels exactly.
A one-sided estimate is biased by half the gradient across the separation. Reach for `one_sided`
only when one side is unusable (a neighbor, a bad column, the edge of the slit), and expect a
small bias when you do.

## `statistic` and outlier rejection

Two more parameters control how the pixels inside the windows are combined. `statistic` chooses
between `"average"` (the default, which handles partial-pixel weights at the window edges) and
`"median"`. Separately, `sigma` applies sigma clipping column by column before the combination,
with a default of 5.0; `sigma=None` disables it.

In [ ]:
for statistic in ("average", "median"):
    flux = (
        Background.two_sided(image, trace, separation=60, width=20, statistic=statistic)
        .bkg_spectrum()
        .flux.value
    )
    print(f"{statistic:>8}: sky at column 512 = {flux[512]:7.2f} e-")

The two statistics agree to within about three electrons, which is unsurprising: with 40 mostly
well-behaved pixels per column, the mean and the median of the sky measure the same thing.

Clipping is the more interesting parameter, and its behavior is easy to misread. On this frame it
changes the sky in **only nine of the 1000 columns**. The background is an average over 40 pixels,
so ordinary noise is already averaged down and there is nothing to reject. Where it does act, it
moves the sky by tens to over a hundred electrons, and the two largest corrections have opposite
signs and opposite causes. A **cosmic ray** in one window inflates that column's background by
about 50 e⁻ until clipping removes it. A **dead pixel**, reading some −3000 e⁻ after bias
subtraction, drags another column's background *down* by nearly 140 e⁻. The largest single
correction on the frame is not a cosmic ray at all.

That is the right mental model for `sigma`. It is not a small global improvement; it is a no-op
almost everywhere and a large correction in a few places you cannot predict, in either direction.
Since it costs nothing, leave the default `sigma=5.0` in place.

Two caveats. First, clipping only cleans pixels inside the *background windows*; it does nothing
about a cosmic ray landing on the trace itself, which is `astroscrappy`'s job on the 2D frame
before any of this. Second, clipping assumes the pixels in a window are drawn from one
distribution. Where that is not true, on a steep illumination gradient or on a frame that has
*not* been rectified so that a sky line's flux genuinely varies across the window, an aggressive
`sigma` can start rejecting real signal. The default of 5.0 is conservative enough to avoid this;
do not lower it without checking.

With clipping enabled, `average` is the better `statistic`: it is less noisy than the median for
well-behaved data and handles fractional pixels at the window edges properly.

## Inspecting the residuals

The separation test asked whether the sky came out *level*. The sharper question is whether it
came out level at every wavelength, because a residual that tracks the sky spectrum is the
signature of a systematic rather than of noise. The test is to split the columns of a blank strip
into those on bright sky lines and those on sky continuum, and to compare the scatter of the
background-subtracted residual in each.

On the rectified frame the sky lines subtract to the photon noise: the residual scatter on lines a
thousand electrons bright is within ten percent of the prediction for averaging 20 strip rows and
40 background pixels. On the continuum the residual is about twice the prediction, the excess being
narrow structure in the sky that a per-column constant cannot follow. The absolute residual is
about the same in both cases, 12 to 14 e⁻, and does not grow with sky brightness.

That independence from sky brightness is what rectification bought, and it is the reason this
chapter comes after Chapter 6. `Background` measures the sky 60 rows from the trace and subtracts
it from the trace's own row, which is only meaningful if both rows hold the same wavelengths in the
same columns. On the raw frame they do not. Chapter 6 measured a drift of 5.6 pixels across the
slit, so on a steep sky line the background model is offset in wavelength from the sky it is
subtracted from, and the line does not cancel.

The signature is specific. Each line leaves a residual shaped like its derivative, positive on one
flank and negative on the other, and the damage grows with separation because the wavelength
mismatch does. The same diagnostic run at separations from 40 to 80 gives a line-to-continuum
residual ratio that climbs from 1.6 to 2.0 on the raw frame and stays flat at about 1.0 on the
rectified one. No choice of `separation`, `width`, `statistic` or `sigma` could have removed that
systematic.

:::{note}
Rectification is not what makes the sky-line residuals *small*; it is what makes them
**independent of where you measure the sky**. On a frame whose tilt is negligible the ratio would
be flat on both frames, and the correction would buy nothing. Chapter 6 ends with a rule of thumb
for telling which case you are in.
:::

## Summary

- `Background` estimates the sky in windows that run **parallel to the trace**, offset by
  `separation` and spanning `width` rows, and reduces each column to a single number. It has no
  model of variation along the slit, so anything left in a window is averaged into the sky, and
  it will not warn you.
- **Read the slit profile before choosing windows.** This frame carries four sources (rows 36,
  117, 135, 311), and the target's wings reach about 60 rows out.
- Separation has a genuine optimum. Too close and the source's own wings inflate the sky: at 10
  rows out the target still contributes half the sky level. Too far and the windows reach a
  neighbor: at separation 95 the lower window swallows the trace at row 36. Anything from roughly
  50 to 85 is defensible here, and the width trades noise against the risk of reaching a neighbor.
- Prefer `two_sided`. The sky differs by about four percent between the two sides of the trace,
  and symmetric averaging cancels a linear gradient exactly. A one-sided estimate is biased by
  half the gradient across the separation, so reach for `one_sided` only when one side is
  unusable.
- `statistic="average"` with the default `sigma=5.0` is a good default. Clipping acts in only nine
  of 1000 columns here, but where it does the correction reaches 140 e⁻, from a cosmic ray or a
  dead pixel in a window. It is cheap insurance, not a substitute for cosmic-ray rejection, and it
  does nothing about a hit on the trace itself.
- **Rectify first.** Subtracting sky measured 60 rows away assumes that row holds the same
  wavelengths as the trace's own. On the raw frame it does not, so every sky line leaves a
  derivative-shaped residual that grows with separation, from about 1.6 to 2.0 times the continuum
  residual. On the rectified frame the ratio is flat at about 1.0, independent of where the sky is
  measured, and the sky lines subtract at photon noise.
- The settings carried forward are **`separation=60, width=20`**.

Next: [Spectral Extraction](08-extraction.ipynb), which collapses the background-subtracted frame
to one dimension.